# InSAR Sentinel-1 + ISCE 端到端流程（合并版）

这个 Notebook 用于替代原有 3 个 Notebook：
1. 输入与裁剪（原 Part1）
2. LSTM 预测与评分（原 Part2）
3. 输出地理编码与 GeoTIFF（原 Part3）

原则：**Notebook 只做说明和少量调度**，详细逻辑都在 `insar_pipeline/*.py` 中。

## 1) 路径配置
请先按你的机器修改路径。

In [ ]:
from pathlib import Path

BASE_INTERFEROGRAM_DIR = Path("/data6/WORKDIR/AmatriceSenDT22/merged/interferograms")
GEOM_REFERENCE_DIR = Path("/data6/WORKDIR/AmatriceSenDT22/merged/geom_reference")
NEXT_DATE = "20160821_20160902"

## 2) 分步执行（推荐调试）

In [ ]:
from insar_pipeline import (
    CropConfig, batch_crop_filt_fine_cor,
    DatasetConfig, build_and_save_dataset,
    TrainingConfig, run_training_and_prediction,
    ScoreConfig, compute_and_save_score,
    OutputConfig, generate_geocoded_outputs,
)

In [ ]:
# Step A: 裁剪 cor + lat/lon
cropped_outputs = batch_crop_filt_fine_cor(
    CropConfig(
        base_path=BASE_INTERFEROGRAM_DIR,
        geom_reference_path=GEOM_REFERENCE_DIR,
        output_base_path=BASE_INTERFEROGRAM_DIR / "cropped",
    )
)
print(f"cropped files: {len(cropped_outputs)}")

In [ ]:
# Step B: 构建训练数据（data.npy / data_std.npy / dates.pkl / geninue*.npy）
dataset_dir = build_and_save_dataset(
    DatasetConfig(
        cropped_dir=BASE_INTERFEROGRAM_DIR / "cropped",
        output_dir=BASE_INTERFEROGRAM_DIR / "cropped",
    )
)
print("dataset_dir:", dataset_dir)

In [ ]:
# Step C: 训练与预测
predict_dir = run_training_and_prediction(
    TrainingConfig(
        dataset_dir=dataset_dir,
        output_dir=BASE_INTERFEROGRAM_DIR / "cropped",
        next_date=NEXT_DATE,
        epochs=15,
    )
)
print("predict_dir:", predict_dir)

In [ ]:
# Step D: 计算 score.npy
score_path = compute_and_save_score(
    ScoreConfig(
        dataset_dir=dataset_dir,
        predict_dir=predict_dir,
    )
)
print("score:", score_path)

In [ ]:
# Step E: 产出 geocode + subset + GeoTIFF
output_files = generate_geocoded_outputs(
    OutputConfig(
        predict_dir=predict_dir,
        lat_file=BASE_INTERFEROGRAM_DIR / "cropped" / "lat_cropped.rdr",
        lon_file=BASE_INTERFEROGRAM_DIR / "cropped" / "lon_cropped.rdr",
    )
)
print(output_files)

## 3) 一键执行（稳定后使用）

In [ ]:
from insar_pipeline import run_full_pipeline

result = run_full_pipeline(
    base_dir=BASE_INTERFEROGRAM_DIR,
    geom_reference_dir=GEOM_REFERENCE_DIR,
    next_date=NEXT_DATE,
)
result

## 4) 后续扩展建议
- 把超参数（如 epochs、batch size、bbox）独立到 `yaml` 配置文件。
- 将训练日志统一输出到 `predict/train.log`。
- 若数据规模进一步增大，可在 `modeling.py` 中引入分块训练/采样策略。